# Doctor-Assistant — Full System Test

This notebook exercises **every routing path** the system knows about using a mixed panel of test scans:

| # | Scenario | Modality | Body part | Expert(s) expected |
|---|---|---|---|---|
| 1 | Normal chest X-ray | X-ray | Chest | chest_xray |
| 2 | Abnormal CXR — Effusion + Nodule | X-ray | Chest | chest_xray |
| 3 | BiomedCLIP backbone (same scan) | X-ray | Chest | chest_xray (BiomedCLIP) |
| 4 | MAIRA-2 grounded reporter | X-ray | Chest | chest_maira2 |
| 5 | CT abdomen — organ volumetry | CT | Abdomen | ct_totalsegmentator |
| 6 | Router fallback — unknown body part | X-ray | UNKNOWN | chest_xray (fallback) |
| 7 | Router miss — no expert registered | MRI | Brain | RoutingError |

**Data sources (no authentication needed):**
- Chest X-rays: `medmnist` ChestMNIST — frontal CXRs at 224px (→ resized to the model's 320px), auto-downloads, CC BY 4.0  
- CT abdomen: synthetic NIfTI with realistic Hounsfield values (no download needed)  
- Scenarios 3 / 4 / 5 are opt-in (large model downloads) — set the flags in cell 5

> **Note on accuracy:** ChestMNIST is a *pre-downsampled benchmark*, not full-resolution clinical imagery. It validates the routing/pipeline plumbing, not diagnostic accuracy. Subtle findings may stay below threshold because detail was lost in MedMNIST's downsampling — for real answers, feed native-resolution DICOM/PNG X-rays.

**Runtime:** L4 GPU recommended. CPU works for everything except MAIRA-2.

## 0 · GPU check

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)')
else:
    print('No GPU — running on CPU (MAIRA-2 will be very slow)')
print(f'PyTorch: {torch.__version__}')

## 1 · Pull codebase

In [ ]:
import os, sys, importlib
!git -C /content/doctor_assistant pull --ff-only
REPO_URL = "https://github.com/Hamza09Hamza/doctor_assistant.git"
REPO_DIR = "/content/doctor_assistant"

if os.path.isdir(REPO_DIR):
    ret = os.system(f"git -C {REPO_DIR} pull --ff-only -q")
    if ret != 0:
        raise RuntimeError("git pull failed — check network.")
else:
    ret = os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
    if ret != 0 or not os.path.isdir(REPO_DIR):
        raise RuntimeError("git clone failed — check network.")

sys.path.insert(0, REPO_DIR)

for mod in ["experts", "routing", "reporting", "pipeline", "core"]:
    if importlib.util.find_spec(mod) is None:
        raise ImportError(f"Module '{mod}' not found — clone may be incomplete.")

print(f"Repo ready.  Contents: {os.listdir(REPO_DIR)}")

## 2 · Install dependencies

In [ ]:
%%capture
# Core stack (always needed)
!pip install -q \
    'monai>=1.3' 'nibabel>=5.2' 'timm>=0.9' \
    'scikit-learn>=1.4' 'scikit-image>=0.22' \
    'pydantic>=2.6' 'Pillow>=10'

# Mixed test data — ChestMNIST (no auth, auto-download)
!pip install -q medmnist

# ── Optional pretrained models — uncomment to enable ────────────────────────
# Scenario 3: BiomedCLIP backbone
# !pip install -q open_clip_torch
# Scenario 4: MAIRA-2 grounded reporter (~7B params)
# !pip install -q 'transformers>=4.46' protobuf sentencepiece
# Scenario 5: TotalSegmentator CT organ segmentation
# !pip install -q TotalSegmentator
# Local LLM report polish (Phi-3 Mini, 4-bit) — needed when RUN_LLM=True.
# Fully local: weights download once from HuggingFace, no API key, no network at inference.
# !pip install -q 'transformers>=4.46' accelerate bitsandbytes

print('Dependencies installed.')

## 3 · Mount Drive (to load the trained checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_PATH = '/content/drive/MyDrive/doctor_assistant/checkpoints/chest_xray/best.pt'
has_checkpoint = os.path.isfile(CKPT_PATH)
print(f'Trained checkpoint on Drive: {has_checkpoint}')
if not has_checkpoint:
    print('  → Will use random weights (wiring test still valid; AUC numbers are random)')

## 4 · Test-scenario flags

Flip the ones you want to run. Scenarios 3-5 each download a large model on first run.

In [ ]:
# ── Which optional experts to include ────────────────────────────────────────
RUN_BIOMEDCLIP   = False   # Scenario 3: swap backbone to BiomedCLIP ViT-B/16
RUN_MAIRA2       = False   # Scenario 4: add MAIRA-2 as a second chest reader
RUN_CT           = False   # Scenario 5: add TotalSegmentator for CT abdomen

# ── Report drafting ──────────────────────────────────────────────────────────
# False → deterministic template reports (fast, no download).
# True  → local LLM (Phi-3 Mini, 4-bit ~2GB) polishes the prose. Fully local,
#         no API key. Requires the LLM deps in cell 3 to be uncommented first.
RUN_LLM          = False

# ── CXR model config (must match what was trained) ───────────────────────────
BACKBONE   = 'timm:densenet121'
IMAGE_SIZE = 320

# Overrides if BiomedCLIP is requested (it has a fixed 224px input)
if RUN_BIOMEDCLIP:
    BACKBONE   = 'biomedclip:'
    IMAGE_SIZE = 224

print(f'Backbone: {BACKBONE}  |  image_size: {IMAGE_SIZE}')
print(f'MAIRA-2: {RUN_MAIRA2}  |  TotalSegmentator: {RUN_CT}  |  LLM: {RUN_LLM}')

## 5 · Download mixed test data

**ChestMNIST** — 112,120 frontal chest X-rays from NIH, pre-downsampled by the MedMNIST
authors (28/64/128/224px). We pull the **224px** test split (22,433 images) so the source
is close to the model's 320px input — upscaling a smaller image can't recover lost detail.
We also generate a synthetic CT NIfTI (realistic Hounsfield values) for the CT routing
scenario.

In [ ]:
import numpy as np
import medmnist
from medmnist import ChestMNIST
from torchvision import transforms as T

# ── Source resolution of the ChestMNIST download ────────────────────────────
# ChestMNIST is NIH ChestX-ray14 PRE-DOWNSAMPLED by the MedMNIST authors. Options:
# 28 / 64 / 128 / 224. Pick the source as close to the model's input (320) as
# possible — upscaling a tiny image CANNOT recover detail the downsample threw away,
# so a 64px source → 320px is a blurry mush the model (correctly) finds nothing in.
#   224 → 320  upscale 1.4×  (best fidelity, ~5 GB download)
#   128 → 320  upscale 2.5×  (good compromise, ~1.6 GB)
#    64 → 320  upscale 5.0×  (fast, but detail is gone — findings get suppressed)
# NOTE: this is a TEST-DATA limitation only. For real answers, feed native-resolution
# DICOM/PNG X-rays so preprocessing downsamples a SHARP source to 320.
CHESTMNIST_SIZE = 224

# source grayscale → resized to IMAGE_SIZE (320), repeated to 3 channels
cxr_transform = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
])
chest_ds = ChestMNIST(split='test', transform=cxr_transform, download=True, size=CHESTMNIST_SIZE)
print(f'ChestMNIST test split: {len(chest_ds):,} images  (source {CHESTMNIST_SIZE}px → {IMAGE_SIZE}px)')
print(f'Label names: {chest_ds.info["label"]}')

# Pull a few specific samples for repeatable scenarios
# ChestMNIST labels: multi-hot vector over 14 pathologies (same NIH labels)
all_labels  = np.array([chest_ds[i][1] for i in range(len(chest_ds))])

# Sample 1: no findings ("normal")
normal_idxs  = np.where(all_labels.sum(axis=1) == 0)[0]
normal_idx   = int(normal_idxs[0])

# Sample 2: at least 2 pathologies co-present
abnormal_idxs = np.where(all_labels.sum(axis=1) >= 2)[0]
abnormal_idx  = int(abnormal_idxs[5])

print(f'Normal sample  #{normal_idx}: labels = {all_labels[normal_idx]}')
print(f'Abnormal sample #{abnormal_idx}: labels = {all_labels[abnormal_idx]}')

In [ ]:
import nibabel as nib

# ── Synthetic CT abdomen NIfTI ─────────────────────────────────────────────
# 256×256×100 volume with Hounsfield values typical of an abdominal CT.
# Liver ~60 HU, spleen ~40 HU, kidney ~30 HU, fat ~-100 HU, air ~-1000 HU.
# TotalSegmentator needs a NIfTI on disk with real voxel spacing.

SYNTH_CT_PATH = '/content/synth_abdomen_ct.nii.gz'

np.random.seed(42)
vol = np.full((256, 256, 100), -100.0, dtype=np.float32)   # fat background

# Air outside body
cx, cy = 128, 128
for z in range(100):
    Y, X = np.ogrid[:256, :256]
    body = (X - cx)**2 + (Y - cy)**2 < 110**2
    vol[:, :, z][~body] = -1000.0

# Liver (right lobe, HU ~40-80)
vol[60:180, 80:200, 30:80] += np.random.uniform(40, 80, (120, 120, 50)).astype(np.float32) + 100
# Spleen (left side, HU ~35-55)
vol[60:130, 50:110, 35:75] += np.random.uniform(35, 55, (70, 60, 40)).astype(np.float32) + 100
# Aorta (central, HU ~200+)
vol[115:145, 115:145, 10:90] += 300.0

affine = np.diag([1.5, 1.5, 2.5, 1.0])  # 1.5mm×1.5mm×2.5mm voxels
img = nib.Nifti1Image(vol, affine)
nib.save(img, SYNTH_CT_PATH)
print(f'Synthetic CT saved: {SYNTH_CT_PATH}  shape={vol.shape}  spacing=(1.5,1.5,2.5)mm')

## 6 · Build the expert panel + pipeline

In [ ]:
import torch
from experts import build_chest_xray_expert, build_default_registry
from routing import ModalityRouter
from reporting import GridZoneLocalizer, Verifier, GuidelineEngine, Reporter
from pipeline import Pipeline
from core.enums import BodyPart, Modality
from data.chest_xray14 import CHESTXRAY14_LABELS
from models.experts import strip_orig_mod

# ── 1. Chest expert (trained weights from Drive, or random for wiring test) ──
chest_expert = build_chest_xray_expert(
    backbone=BACKBONE, image_size=IMAGE_SIZE, pretrained=not RUN_BIOMEDCLIP
).to(device).eval()

if has_checkpoint:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    chest_expert.load_state_dict(strip_orig_mod(ckpt['model']))
    print(f"Chest expert: loaded from checkpoint  "
          f"(epoch {ckpt['epoch']}, AUC {ckpt.get('best_metric', float('nan')):.4f})")
else:
    print('Chest expert: random weights (no checkpoint found on Drive)')

# ── 2. Assemble the expert registry ──────────────────────────────────────────
registry = build_default_registry(
    chest_expert=chest_expert,
    include_maira2=RUN_MAIRA2,
    include_ct=RUN_CT,
)
niches = sorted((m.value, b.value) for (m, b) in registry._by_niche)
print(f'Registered niches: {niches}')

# ── 3. Reporter — local LLM (Phi-3) only when RUN_LLM=True, else template ──────
reporter = Reporter(auto_llm=RUN_LLM)
print(f'Reporter LLM: {"Phi-3 Mini (local, 4-bit)" if reporter.llm else "template only"}')

# ── 4. Build the orchestrator ─────────────────────────────────────────────────
# strict=False so we can test the fallback path (scenario 6)
router = ModalityRouter(registry, strict=False)
pipe   = Pipeline(
    router,
    reporter=reporter,
    verifier=Verifier(known_labels=list(CHESTXRAY14_LABELS)),
    guidelines=GuidelineEngine(),
    thresholds=0.4,          # lower threshold to surface more findings in the demo
    localizer=GridZoneLocalizer(),
)
print('Pipeline ready.')

## 7 · Helper: run one scan and print the full result

In [ ]:
from core.types import Scan, ScanMetadata
from routing.router import RoutingError
import traceback

RESULTS = []   # collect for the summary table at the end

def run_scenario(label, data_tensor, modality, body_part, source_path=None, expect_error=False):
    print(f'\n{"="*72}')
    print(f'SCENARIO: {label}')
    print(f'  Modality={modality.value}  BodyPart={body_part.value}')
    print(f'{"="*72}')

    scan = Scan(
        data=data_tensor.to(device),
        meta=ScanMetadata(
            modality=modality,
            body_part=body_part,
            source_path=source_path,
        )
    )

    try:
        result = pipe.analyze_scan(scan)
    except RoutingError as e:
        print(f'  [RoutingError — expected={expect_error}]  {e}')
        RESULTS.append({
            'scenario': label,
            'modality': modality.value,
            'body_part': body_part.value,
            'experts': '— (routing error)',
            'findings': '—',
            'urgency': '—',
            'verifier': '—',
            'generator': '—',
        })
        return

    present = [f for f in result.findings if f.present]
    print(f'  Routed to  : {result.experts}')
    print(f'  Findings   : {[f.label for f in present] or "[none above threshold]"}')

    # Raw top scores (diagnostic) — what the model actually predicted, BEFORE the
    # threshold filter drops sub-threshold labels. Lets you tell "model saw nothing"
    # (scores ~0.05) apart from "just under the line" (scores ~0.38).
    raw = {}
    for p in result.predictions:
        raw.update(getattr(p, 'class_probs', {}) or {})
    if raw:
        top5 = sorted(raw.items(), key=lambda kv: kv[1], reverse=True)[:5]
        print('  Top scores : ' + ', '.join(f'{k}={v:.2f}' for k, v in top5))

    print(f'  Triage     : {result.triage_urgency.name}')
    print(f'  Report gen : {result.report.generator}')
    verif = result.verification
    verif_str = f'{"PASS" if verif.ok else "FAIL"} ({verif.grounded_fraction*100:.0f}% grounded)'
    print(f'  Verifier   : {verif_str}')
    if verif.flags:
        for flag in verif.flags:
            print(f'    ✗ {flag}')
    print()
    print(result.report.to_text())
    if result.recommendations:
        print('\nRECOMMENDATIONS:')
        for r in result.recommendations:
            print(f'  [{r.urgency.name}] {r.label}: {r.text}')

    RESULTS.append({
        'scenario': label,
        'modality': modality.value,
        'body_part': body_part.value,
        'experts': ', '.join(result.experts),
        'findings': ', '.join(f.label for f in present) or 'none',
        'urgency': result.triage_urgency.name,
        'verifier': verif_str,
        'generator': result.report.generator,
    })

print('Helper ready.')

## 8 · Scenario 1 — Normal chest X-ray
A study where no pathology reaches the threshold. The pipeline should report no acute finding and ROUTINE urgency.

In [ ]:
img_normal, label_normal = chest_ds[normal_idx]
run_scenario(
    label='1 · Normal chest X-ray',
    data_tensor=img_normal,
    modality=Modality.XRAY,
    body_part=BodyPart.CHEST,
)

## 9 · Scenario 2 — Abnormal chest X-ray (multiple findings)
A study with 2+ co-present pathologies. We expect a PROMPT or URGENT triage depending on which labels fire.

In [ ]:
img_abnormal, label_abnormal = chest_ds[abnormal_idx]
true_labels_abnormal = [
    list(chest_ds.info['label'].values())[i]
    for i, v in enumerate(label_abnormal.squeeze()) if v > 0
]
print(f'Ground-truth labels for this sample: {true_labels_abnormal}')

run_scenario(
    label='2 · Abnormal CXR — multiple pathologies',
    data_tensor=img_abnormal,
    modality=Modality.XRAY,
    body_part=BodyPart.CHEST,
)

## 10 · Scenario 3 — BiomedCLIP backbone  *(opt-in, RUN_BIOMEDCLIP=True)*

Shows that swapping the backbone is a single toggle — the rest of the pipeline is identical.
Skipped when `RUN_BIOMEDCLIP=False`.

In [ ]:
if RUN_BIOMEDCLIP:
    # BiomedCLIP expert is already the active `chest_expert` when backbone='biomedclip:'
    run_scenario(
        label='3 · BiomedCLIP backbone (same scan)',
        data_tensor=img_abnormal,
        modality=Modality.XRAY,
        body_part=BodyPart.CHEST,
    )
else:
    print('Scenario 3 skipped (RUN_BIOMEDCLIP=False). '
          'Set RUN_BIOMEDCLIP=True in cell 5 and re-run from there.')

## 11 · Scenario 4 — MAIRA-2 grounded reporter  *(opt-in, RUN_MAIRA2=True)*

MAIRA-2 runs **alongside** the trained classifier — both are registered under `(XRAY, CHEST)`,
so the router returns both and the orchestrator pools their findings. The grounded sentences
from MAIRA-2 are parsed into canonical `Finding`s with location boxes.

In [ ]:
if RUN_MAIRA2:
    run_scenario(
        label='4 · MAIRA-2 grounded reporter',
        data_tensor=img_abnormal,
        modality=Modality.XRAY,
        body_part=BodyPart.CHEST,
    )
else:
    print('Scenario 4 skipped (RUN_MAIRA2=False). '
          'Uncomment transformers in cell 3, set RUN_MAIRA2=True, re-run from cell 3.')

## 12 · Scenario 5 — CT abdomen, organ volumetry  *(opt-in, RUN_CT=True)*

TotalSegmentator segments the synthetic NIfTI, returns liver/spleen/kidney volumes in mL.
The measurements are real physical units (voxel count × voxel spacing / 1000).

In [ ]:
if RUN_CT:
    import nibabel as nib
    ct_img = nib.load(SYNTH_CT_PATH)
    ct_tensor = torch.as_tensor(
        np.asarray(ct_img.dataobj).astype(np.float32)
    ).unsqueeze(0)   # add channel dim (1, D, H, W)

    run_scenario(
        label='5 · CT abdomen — TotalSegmentator organ volumetry',
        data_tensor=ct_tensor,
        modality=Modality.CT,
        body_part=BodyPart.ABDOMEN,
        source_path=SYNTH_CT_PATH,   # TotalSegmentator reads the file directly
    )
else:
    print('Scenario 5 skipped (RUN_CT=False). '
          'Uncomment TotalSegmentator in cell 3, set RUN_CT=True, re-run from cell 3.')

## 13 · Scenario 6 — Router fallback (UNKNOWN body part)

When the body part isn't known, the router tries to find *any* expert for the modality.
Here `(XRAY, UNKNOWN)` falls back to the chest expert because it handles X-ray.

In [ ]:
run_scenario(
    label='6 · Router fallback — XRAY with UNKNOWN body part',
    data_tensor=img_abnormal,
    modality=Modality.XRAY,
    body_part=BodyPart.UNKNOWN,
)

## 14 · Scenario 7 — Router miss (no expert registered for MRI Brain)

No brain MRI expert has been registered yet. The router (strict=False) exhausts all
fallbacks and raises `RoutingError`, which `run_scenario` catches and records.

In [ ]:
# Synthetic MRI volume (3D, single channel)
mri_tensor = torch.rand(1, 128, 128, 64)  # (C, H, W, D)

run_scenario(
    label='7 · Router miss — MRI Brain (no expert registered)',
    data_tensor=mri_tensor,
    modality=Modality.MRI,
    body_part=BodyPart.BRAIN,
    expect_error=True,
)

## 15 · Visualise — Grad-CAM on the two CXR scenarios

In [ ]:
import matplotlib.pyplot as plt
from explainability import GradCAM
from reporting import findings_from_classification
from core.types import Scan, ScanMetadata

def show_gradcam(title, img_tensor):
    scan = Scan(
        data=img_tensor.to(device),
        meta=ScanMetadata(modality=Modality.XRAY, body_part=BodyPart.CHEST),
    )
    pred  = chest_expert.predict(scan)
    cam   = GradCAM(chest_expert)
    hmaps = cam.for_labels(scan.data)
    findings = findings_from_classification(
        pred.class_probs, thresholds=0.4, localizer=GridZoneLocalizer()
    )
    present = [f for f in findings if f.present][:3]

    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    gray   = img_np[:, :, 0]

    n = len(present) + 1
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    axes[0].imshow(gray, cmap='gray'); axes[0].set_title(f'{title}\n(input)'); axes[0].axis('off')
    for ax, f in zip(axes[1:], present):
        hm = hmaps.get(f.label)
        ax.imshow(gray, cmap='gray')
        if hm is not None:
            ax.imshow(hm.numpy(), alpha=0.5, cmap='jet', vmin=0, vmax=1)
        ax.set_title(f'{f.label}\np={f.probability:.2f}  {f.location or ""}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Top findings: {[(f.label, f.laterality, f.location) for f in present]}')

from reporting import GridZoneLocalizer
show_gradcam('Normal scan', img_normal)
show_gradcam('Abnormal scan', img_abnormal)

## 16 · Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 160)
print(df.to_string(index=False))

## 17 · Smoke test — run the repo's wiring check

Runs the same parser-core + routing checks that CI would run, but now with the real repo pulled from GitHub.

In [ ]:
!python {REPO_DIR}/scripts/smoke_system.py